In [1]:
# Step 1 - Mount Drive correctly
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys

sys.path.append('/content/drive/MyDrive/NLP_project')

In [3]:
import os
import pandas as pd

# 1. Define paths
project_path = '/content/drive/MyDrive/NLP_project'

# 3. Find and process all CSV files
for root, dirs, files in os.walk(project_path):
    for file in files:
        if file:
            file_path = os.path.join(root, file)
            print(f"Processing: {file_path}")

Processing: /content/drive/MyDrive/NLP_project/ISOT_Fake.csv
Processing: /content/drive/MyDrive/NLP_project/ISOT_True.csv
Processing: /content/drive/MyDrive/NLP_project/WELFake.csv
Processing: /content/drive/MyDrive/NLP_project/roberta_config.yaml
Processing: /content/drive/MyDrive/NLP_project/processed_train_df.csv
Processing: /content/drive/MyDrive/NLP_project/processed_val_df.csv
Processing: /content/drive/MyDrive/NLP_project/processed_test_df.csv


In [4]:
import os
import pandas as pd
import numpy as np
import torch
import logging
from transformers import (
        RobertaTokenizer,
        RobertaForSequenceClassification,
        Trainer,
        TrainingArguments
)
from torch.utils.data import Dataset

In [5]:

# paths
train_data = "/content/drive/MyDrive/NLP_project/WELFake.csv"
isot_true = "/content/drive/MyDrive/NLP_project/ISOT_True.csv"
isot_fake = "/content/drive/MyDrive/NLP_project/ISOT_Fake.csv"
output_dir = "/content/drive/MyDrive/NLP_project/artifacts/roberta_model"
logging_dir = "/content/drive/MyDrive/NLP_project/logs/"

# Model & Tokenizer Specifications
# model
model_name_or_path = "roberta-base"
num_labels = 2
max_length = 256  # Sliced length to balance context retention and GPU VRAM capacity

# Training Hyperparameters
# training_args
num_train_epochs = 3
per_device_train_batch_size = 16
per_device_eval_batch_size = 16
warmup_steps = 500
weight_decay = 0.01
learning_rate = 2e-5  # Safe, standard starting learning rate for fine-tuning transformers
evaluation_strategy = "epoch"
save_strategy = "epoch"
load_best_model_at_end = True
metric_for_best_model = "f1"
fp16 = True  # Set to true if your GPU supports mixed precision training to maximize speed

In [8]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class FakeNewsDataset(Dataset):
    def __init__(self, texts: list, labels: list, tokenizer, max_len: int):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        # Labels must be numeric {0,1}. Keep them as-is.
        label = self.labels[item]


        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }
@staticmethod
def prepare_transform_pipeline():
    logger.info(f"Initializing Tokenizer: {model_name_or_path}")
    tokenizer = RobertaTokenizer.from_pretrained(model_name_or_path)
    return tokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
def compute_metrics(eval_pred):
    """
    Computes precision, recall, f1, and accuracy for sequence classification.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='binary'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

if __name__ == "__main__":
    # Initialize Pipeline Configs and Tokenizer
    tokenizer = prepare_transform_pipeline()

    # Ingest the clean CSV data frames directly
    logger.info("Ingesting clean datasets for Transformer Optimization...")
    train_df = pd.read_csv("/content/drive/MyDrive/NLP_project/processed_train_df.csv").dropna(subset=['text']).reset_index(drop=True)
    val_df = pd.read_csv("/content/drive/MyDrive/NLP_project/processed_val_df.csv").dropna(subset=['text']).reset_index(drop=True)

    # Format into custom PyTorch Dataset wrappers
    logger.info("Encoding text data fields into PyTorch tensor structures...")
    train_dataset = FakeNewsDataset(
        texts=train_df['text'].tolist(),
        labels=train_df['label'].tolist(),
        tokenizer=tokenizer,
        max_len=max_length
    )

    val_dataset = FakeNewsDataset(
        texts=val_df['text'].tolist(),
        labels=val_df['label'].tolist(),
        tokenizer=tokenizer,
        max_len=max_length
    )

    # Initialize Pre-trained RoBERTa Architecture
    logger.info("Downloading pre-trained RoBERTa sequence classification weights...")
    model = RobertaForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=num_labels
    )

    # Build Training Arguments from YAML parameters
    training_arguments = TrainingArguments(
        output_dir = output_dir,
        logging_dir = logging_dir,
        num_train_epochs = num_train_epochs,
        per_device_train_batch_size = per_device_train_batch_size,
        per_device_eval_batch_size = per_device_eval_batch_size,
        warmup_steps = warmup_steps,
        weight_decay = weight_decay,
        learning_rate = float(learning_rate),
        eval_strategy = evaluation_strategy,
        save_strategy = save_strategy,
        load_best_model_at_end = load_best_model_at_end,
        metric_for_best_model = metric_for_best_model,
        fp16 = fp16,
        logging_steps = 100
    )

    # Instantiate Hugging Face Trainer Instance
    trainer = Trainer(
        model=model,
        args=training_arguments,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # 7. Execute Fine-Tuning Optimization Execution Run
    logger.info("Launching Transformer training loop on active hardware acceleration device...")
    trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.016266,0.013724,0.997343,0.997593,0.997497,0.997689
2,0.010986,0.018968,0.996386,0.996736,0.993684,0.999807
3,0.006081,0.011492,0.998512,0.998653,0.997694,0.999615


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]